# Part 3: NLP Text Classification

## Objective

The objective of this project is to classify customer support messages into positive, neutral, and negative sentiment categories using NLP techniques and deep learning models.

In [ ]:
# Import Libraries

In [ ]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
import re

from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from sklearn.linear_model import LogisticRegression

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import LSTM

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

nltk.download('stopwords')

print("Libraries Imported Successfully")

# Dataset Loading

In [ ]:
# =========================================================
# LOAD DATASET
# =========================================================

df = pd.read_csv(
    "data/customer_support_text_classification.csv"
)

print("Dataset Shape:", df.shape)

df.head()

# Dataset Exploration

print(df.columns)

print(df.info())

# Sentiment Distribution

In [ ]:
import os

os.makedirs("results", exist_ok=True)

plt.figure(figsize=(6,4))

sns.countplot(x=df['sentiment_label'])

plt.title("Sentiment Distribution")

plt.savefig("results/sentiment_distribution.png")

plt.show()

In [ ]:
# Text Cleaning

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z ]', '', text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    return " ".join(words)

df['cleaned_text'] = df['customer_message'].apply(clean_text)

df[['customer_message', 'cleaned_text']].head()

# TF-IDF Vectorization

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(
    df['cleaned_text']
).toarray()

encoder = LabelEncoder()

y = encoder.fit_transform(
    df['sentiment_label']
)

print("Feature Shape:", X.shape)

# Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Shape:", X_train.shape)

print("Testing Shape:", X_test.shape)

# Baseline NLP Model

In [ ]:
model = LogisticRegression()

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(classification_report(y_test, predictions))

# Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, predictions)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("Confusion Matrix")

plt.savefig("results/confusion_matrix.png")

plt.show()

# Deep Learning NLP Model using LSTM

In [ ]:
tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(df['cleaned_text'])

sequences = tokenizer.texts_to_sequences(
    df['cleaned_text']
)

X_seq = pad_sequences(
    sequences,
    maxlen=100
)

X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(
    X_seq,
    y,
    test_size=0.2,
    random_state=42
)

deep_model = Sequential([

    Embedding(
        5000,
        64,
        input_length=100
    ),

    LSTM(64),

    Dense(
        32,
        activation='relu'
    ),

    Dense(
        3,
        activation='softmax'
    )

])

deep_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = deep_model.fit(
    X_train_seq,
    y_train_seq,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

# Accuracy Curve

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    history.history['accuracy'],
    label='Training Accuracy'
)

plt.plot(
    history.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.title("Model Accuracy")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend()

plt.savefig("results/accuracy_curve.png")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 3, figsize=(18,5))

img1 = mpimg.imread('results/accuracy_curve.png')
img2 = mpimg.imread('results/confusion_matrix.png')
img3 = mpimg.imread('results/sentiment_distribution.png')

axes[0].imshow(img1)
axes[0].axis('off')
axes[0].set_title("Accuracy Curve")

axes[1].imshow(img2)
axes[1].axis('off')
axes[1].set_title("Confusion Matrix")

axes[2].imshow(img3)
axes[2].axis('off')
axes[2].set_title("Sentiment Distribution")

plt.tight_layout()

plt.savefig('results/model_evaluation.png')

plt.show()